In [18]:
# Importing dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix, classification_report
from sklearn.model_selection import GridSearchCV

In [3]:
Dataset_selected = pd.read_csv("Dataset_selected.csv")
Dataset_selected.head()

,radius_mean,radius_worst,concavity_mean,concave points_mean,concavity_worst,concave points_worst,compactness_mean,compactness_worst,texture_mean,texture_worst,radius_se,diagnosis
0,2.943913,3.272606,0.200600,0.120979,0.430225,0.2654,0.171614,0.334015,10.38,17.33,0.391365,M
1,3.071303,3.257712,0.076997,0.063554,0.195896,0.1860,0.070433,0.146640,17.77,23.41,0.307856,M
2,3.029650,3.201526,0.153273,0.107641,0.316152,0.2430,0.129546,0.264616,21.25,25.53,0.366602,M
3,2.519308,2.766948,0.178785,0.091059,0.420612,0.2575,0.171614,0.334015,20.38,26.50,0.291382,M
4,3.058237,3.158701,0.153637,0.090383,0.290033,0.1625,0.111103,0.157850,14.34,16.67,0.369540,M


In [5]:
# Separating Features
X = Dataset_selected.iloc[:, :-1]
y = Dataset_selected.iloc[:, -1]

In [8]:
# Encoding Feature Variable 
le = LabelEncoder()
y = le.fit_transform(y)

is_numpy = isinstance(y, np.ndarray)
print(is_numpy)

True


In [9]:
# Spliting and Feature Scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)  # Spliting into train set

print(X_train.shape)
print(y_train.shape)

(455, 11)
(455,)


In [10]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# Converting to numpy array
X_train = X_train
X_test = X_test

# Model Building

In [13]:
# Trining multiple model
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced'),
    "Random Forest": RandomForestClassifier(),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier()
}


results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred)
    }

# Display results
for model, metrics in results.items():
    print(f"\n{model}")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")


Logistic Regression
Accuracy: 0.9649
F1 Score: 0.9535
Recall: 0.9535

Random Forest
Accuracy: 0.9561
F1 Score: 0.9412
Recall: 0.9302

SVM
Accuracy: 0.9649
F1 Score: 0.9535
Recall: 0.9535

KNN
Accuracy: 0.9649
F1 Score: 0.9524
Recall: 0.9302


In [16]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[70  1]
 [ 3 40]]


In [25]:
# Tunning the Model
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear', 'lbfgs']
}

grid = GridSearchCV(LogisticRegression(class_weight='balanced'),
                    param_grid, cv=5)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)

Best Params: {'C': 1, 'solver': 'lbfgs'}


In [24]:
# Traning final model
final_model = LogisticRegression(C=1, solver='lbfgs', class_weight='balanced')

final_model.fit(X_train, y_train)

# Predicting on test set
y_pred = final_model.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


print(f"\n{'='*150}")
# print('''“Hyperparameter tuning using GridSearchCV identified the optimal Logistic Regression model with C = 0.1 and solver = ‘lbfgs’. The lower C value introduces stronger regularization, improving generalization and reducing overfitting. This optimized model achieved the best balance between recall and precision, making it suitable for medical diagnosis.”
# ''')


              precision    recall  f1-score   support

           0       0.97      0.97      0.97        71
           1       0.95      0.95      0.95        43

    accuracy                           0.96       114
   macro avg       0.96      0.96      0.96       114
weighted avg       0.96      0.96      0.96       114

[[69  2]
 [ 2 41]]

